# Plotly in Notebook Environment


This notebook will act as a starter if you want to create visualizations in a notebook environment before putting in dash app or for an easier dev environment

In [ ]:
import psycopg2
import pandas as pd 
import sqlalchemy as salc
import json
import os
import datetime
#import dash
import plotly.express as px
#from jupyter_dash import JupyterDash
#from dash import dcc
#from dash import html
#from dash.dependencies import Input, Output
import datetime as dt
import plotly

with open("../augur_creds.json") as config_file:
    config = json.load(config_file)

In [ ]:
database_connection_string = 'postgresql+psycopg2://{}:{}@{}:{}/{}'.format(config['user'], config['password'], config['host'], config['port'], config['database'])

dbschema='augur_data'
engine = salc.create_engine(
    database_connection_string,
    connect_args={'options': '-csearch_path={}'.format(dbschema)})

This will allow you to get specific repo_ids. A few are listed for ease of use

In [ ]:
repo_urls = ['https://github.com/ansible/ansible','https://github.com/pulp/pulp-infra-ansible'
             ,'https://github.com/agroal/agroal','https://github.com/chaoss/augur']

url_query = str(repo_urls)
url_query = url_query[1:-1]

repo_query = salc.sql.text(f"""
        SET SCHEMA 'augur_data';
        SELECT DISTINCT
            r.repo_id,
            r.repo_name
        FROM
            repo r
        JOIN repo_groups rg 
        ON r.repo_group_id = rg.repo_group_id
        WHERE
            r.repo_git in({url_query})
        """)

t = engine.execute(repo_query)
results = t.all()
repo_ids = [ row[0] for row in results]
repo_names = [ row[1] for row in results]
print(repo_ids)
print(repo_names)

Below is the query used in the callback. Can copy and paste any query here

In [ ]:
repo_statement = str([32983])
repo_statement = repo_statement[1:-1]

query = salc.sql.text(f"""
                SELECT
                    r.repo_name,
                    c.cmt_commit_hash AS commits,
                    c.cmt_id AS file, 
                    c.cmt_added AS lines_added,
                    c.cmt_removed AS lines_removed,
                    c.cmt_author_date AS date
                FROM
                    repo r
                JOIN commits c 
                ON r.repo_id = c.repo_id
                WHERE
                    c.repo_id in({repo_statement})
                """)
df = pd.read_sql(query, con=engine)

df = df.reset_index()
df.drop("index", axis=1, inplace=True)

Below is the for generating the plotly graph. Any variable inputs from dropdowns or other similar componets I have hardcoded for ease 

In [ ]:
#hard coded in
interval = 86400000

In [ ]:
# reset index to be ready for plotly
df = df.reset_index()

#helper values for building graph 
today = dt.date.today()
x_r = []
x_name = "Year"
hover = "Year: %{x|%Y}"

#graph input values based on date interval selection
if interval == 86400000: #if statement for days
    x_r = [str(today-dt.timedelta(weeks=4)),str(today)]
    x_name = "Day"
    hover = "Day: %{x|%b %d, %Y}"
elif interval == 604800000: #if statmement for weeks 
    x_r = [str(today-dt.timedelta(weeks=30)),str(today)]
    x_name = "Week"
    hover = "Week: %{x|%b %d, %Y}"
elif interval =='M1': #if statement for months
    x_r = [str(today-dt.timedelta(weeks=104)),str(today)]
    x_name = "Month"
    hover = "Month: %{x|%b %Y}"

#graph geration
if(df is not None):
    fig = px.histogram(df, x="date",range_x=x_r,labels={'x':x_name, 'y':'Commits'})
    fig.update_traces(xbins_size=interval , hovertemplate =hover + "<br>Commits: %{y}<br>" )
    fig.update_xaxes(showgrid=True, ticklabelmode="period", dtick=interval,rangeslider_yaxis_rangemode="match" )
    fig.update_layout(
        title={'text':"Commits Over Time",
                  'font':{'size':28},'x':0.5,'xanchor':'center'},
    xaxis_title=x_name,
    yaxis_title="Number of Commits")
    fig.show()